<a href="https://colab.research.google.com/github/alejolion57-stack/Portafolio_Alejandro-Leon/blob/main/Consumo_de_medios_Colombia_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Consumo de medios en Colombia 2026**
--
---

En este cuaderno limpiamos los datos de los data set`indicadores_raw.csv` y `plataformas_raw.csv` para usar como insumo en la construcción y nanálisis utilizando Power Bi.

**Procedimiento:**
1. Carga de datos brutos
2. Explorar qué está mal en estos datos
3. Limpieza de datos
4. Crear tabla de fuentes
5. Armar la tabla principal y guardar
6. Limpiar la tabla de plataformas
7. Control de calidad final
8. Descargar los archivos limpios

Importante: Este notebook se desarrolla con fines de aprendizaje y sus datos deben ser corroborados en caso de usarse como fuente de referencia real.


In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

CANDIDATAS = [Path("indicadores_raw.csv"), Path("plataformas_raw.csv"), Path(".")]
RAW = next(p for p in CANDIDATAS if (p / "indicadores_raw.csv").exists())
CLEAN = RAW.parent / "clean" if RAW.name == "raw" else Path("clean")
CLEAN.mkdir(parents=True, exist_ok=True)

print("Leyendo archivos desde:", RAW.resolve())
print("Guardaré los limpios en:", CLEAN.resolve())

Leyendo archivos desde: /content
Guardaré los limpios en: /content/clean


## Paso 1. Cargar los datos en bruto

Leemos el CSV con `dtype=str`: le decimos a pandas que trate **todo como texto**. Así no "adivina" mal los números (por ejemplo, `"10,49"` con coma decimal) antes de que nosotros los limpiemos.
`keep_default_na=False` evita que convierta `N/D` o celdas vacías en nulos sin avisarnos.

In [2]:
df = pd.read_csv(RAW / "indicadores_raw.csv", dtype=str, keep_default_na=False)
print("Filas:", len(df), "| Columnas:", df.shape[1])
df.head(8)

Filas: 98 | Columnas: 9


,Categoria,Subcategoria,Indicador,Valor,Periodo,Fuente,Tipo de fuente,URL,Notas
0,DEMOGRAFÍA,Población,Población total,"53,6 millones de personas",oct-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,Estimación ONU
1,Demografia,Población,Crecimiento anual de la población,1%,dic-2024 a dic-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,Equivale a +532 mil personas
2,Demografía,Población,Población urbana,83 % de la población,oct-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,
3,Demografía,Población,Población rural,17% de la población,oct-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,
4,DEMOGRAFÍA,Sexo,Población femenina,"50,7% de la población",oct-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,Solo categorías femenino/masculino
5,Demografia,Sexo,Población masculina,"49,3 % de la población",oct-2025,DataReportal / Kepios con datos de la ONU,reporte privado,https://datareportal.com/reports/digital-2026-...,-
6,Demografía,Edad,Edad mediana,"32,5 años",oct-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,
7,Demografía,Edad,Población de 0-4 años,"6,6% de la población",oct-2025,DataReportal / Kepios con datos de la ONU,Reporte privado,https://datareportal.com/reports/digital-2026-...,Puede no sumar 100 % por redondeo


## 🔎Paso 2. Explorar qué está mal en estos datos

Antes de limpiar hay que **mirar**. Revisamos tres columnas problemáticas:
- `Categoria`: debería tener 4 valores, pero ¿cuántos hay realmente?
- `Valor`: viene como texto con unidades.
- `Periodo`: viene escrito de muchas formas.

In [3]:
print("Categorías distintas:", df["Categoria"].nunique(), "(deberían ser 4)")
df["Categoria"].value_counts()

Categorías distintas: 15 (deberían ser 4)


,count
Categoria,
Consumo de medios,12
Economia,9
ECONOMÍA,9
Economía,8
Economía,7
Tecnología,6
TECNOLOGÍA,6
Tecnologia,6
Tecnología,6


In [4]:
# Ejemplos de la columna Valor: número, unidad y base mezclados en un solo texto
df["Valor"].sample(10, random_state=1).tolist()

['33 % de oyentes',
 '3 pp',
 '6,75%',
 '6,74%',
 '60% de encuestados',
 '8,6 %',
 '9,5 %',
 '-5 pp',
 '191,53 Mbps',
 '209 municipios']

In [5]:
# Periodos tal como vienen
df["Periodo"].drop_duplicates().tolist()

['oct-2025',
 'dic-2024 a dic-2025',
 'ago-2026',
 'ago-2025',
 'jul-2026',
 'sep-2026',
 'sep-2025',
 'ene-sep 2026',
 'T2 2026',
 'S1 2026',
 '2026',
 '30/09/2026',
 'ene-feb 2026',
 'oct-2024 a oct-2025',
 'T1 2026',
 'may-2026',
 'dic-2025',
 '2026 vs 2025',
 '2025',
 'S1 2025',
 'N/D (publicado 22/09/2026)']

## 🧼Paso 3. Limpieza de los datos
---
**3.1. Limpieza de texto: espacios, mayúsculas y tildes**

---
Hay valores como `' Economía '` (con espacios), `ECONOMÍA` (mayúsculas) y `Economia` (sin tilde). Para el computador son **cuatro textos distintos**.

Lo arreglamos en dos pasos:
1. Quitar espacios sobrantes en **todas** las columnas.
2. Comparar las categorías sin tildes y en minúscula, y convertirlas al nombre correcto.

In [6]:
def sin_tildes(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

# 1) Quitar espacios al inicio, al final y dobles
for col in df.columns:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

# 2) Unificar categorías
CATEGORIAS = {"demografia": "Demografía", "economia": "Economía",
              "tecnologia": "Tecnología", "consumo de medios": "Consumo de medios"}
df["Categoria"] = df["Categoria"].map(lambda c: CATEGORIAS[sin_tildes(c).lower()])

# 3) Tipo de fuente: primera letra en mayúscula
df["Tipo de fuente"] = df["Tipo de fuente"].str[0].str.upper() + df["Tipo de fuente"].str[1:]

print("Categorías ahora:", df["Categoria"].nunique())
df["Categoria"].value_counts()

Categorías ahora: 4


,count
Categoria,
Economía,33
Tecnología,24
Consumo de medios,24
Demografía,17


---
**3.2. Buscar y eliminar duplicados**

---

Busco duplicados **después** de limpiar el texto. Una fila con un espacio extra no se veía como repetida, pero ahora sí.

Primero las **miramos** (`keep=False` muestra todas las copias) y luego las eliminamos.

In [7]:
print("Filas duplicadas:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("Indicador")[["Categoria", "Indicador", "Valor", "Periodo"]]

Filas duplicadas: 3


,Categoria,Indicador,Valor,Periodo
42,Economía,Crecimiento: actividades artísticas y de entre...,"9,5 %",T2 2026
97,Economía,Crecimiento: actividades artísticas y de entre...,"9,5 %",T2 2026
20,Economía,Inflación anual (IPC),"6,03 %",jul-2026
61,Economía,Inflación anual (IPC),"6,03 %",jul-2026
4,Demografía,Población femenina,"50,7% de la población",oct-2025
30,Demografía,Población femenina,"50,7% de la población",oct-2025


In [8]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Filas antes: {antes} | después: {len(df)} | eliminadas: {antes - len(df)}")

Filas antes: 98 | después: 95 | eliminadas: 3


---
**3.3. Nulos escritos como texto**

---

En la columna `Notas` los vacíos vienen escritos de varias formas: `N/D`, `-`, `s/d`... Para pandas son **texto**, no vacíos. Los convertimos en nulos reales.

In [9]:
NULOS = {"", "N/D", "n/d", "-", "s/d", "n/a"}
df["Notas"] = df["Notas"].where(~df["Notas"].isin(NULOS), None)
print("Filas con nota:", df["Notas"].notna().sum(), "| sin nota:", df["Notas"].isna().sum())

Filas con nota: 47 | sin nota: 48


---
**3.4. Separar el valor en número, unidad y base**

---
La columna `Valor` mezcla todo en un texto: `"53,6 millones de personas"`, `"9,4 %"`, `"$1.750.905"`. No se puede graficar ni sumar así.

La separamos en tres columnas:
- `valor`: el número (convertido del formato español: `1.750.905` → 1750905 y `10,49` → 10.49),
- `unidad`: `%`, millones, COP, Mbps...
- `base`: sobre qué es el porcentaje ("de la población", "de encuestados").

In [10]:
def numero_es(txt: str) -> float:
    """Convierte formato español ('1.750.905', '10,49') a número."""
    return float(txt.replace(".", "").replace(",", "."))


def parse_valor(txt: str):
    m = re.fullmatch(r"([+-]?)\s*(\$?)\s*([\d.]+(?:,\d+)?)\s*(.*)", txt)
    if not m:
        raise ValueError(f"No se pudo leer el valor: {txt!r}")
    signo, dolar, num, resto = m.groups()
    valor = numero_es(num) * (-1 if signo == "-" else 1)
    r, base = resto.lower(), ""
    if dolar or r == "cop":
        unidad = "COP"
    elif r.startswith("%"):
        unidad, base = "%", resto[1:].strip()
    elif r.startswith("millones de cop"):
        unidad = "millones de COP"
    elif r.startswith("millones de personas"):
        unidad = "millones de personas"
    elif r.startswith("millones"):
        unidad = "millones"
    elif r in ("pp", "puntos porcentuales"):
        unidad = "puntos porcentuales"
    elif r.startswith("puesto"):
        unidad = "puesto (de 180)"
    elif r.startswith("accesos por"):
        unidad = "accesos por 100 hab."
    elif r in ("mbps", "años", "municipios"):
        unidad = {"mbps": "Mbps"}.get(r, r)
    else:
        raise ValueError(f"Unidad desconocida: {resto!r} en {txt!r}")
    return valor, unidad, base


# Probamos la función con 4 ejemplos antes de aplicarla a toda la tabla
for ejemplo in ["9,4 %", "$1.750.905", "53,6 millones de personas", "-0,02%"]:
    print(f"{ejemplo!r:32} ->", parse_valor(ejemplo))

'9,4 %'                          -> (9.4, '%', '')
'$1.750.905'                     -> (1750905.0, 'COP', '')
'53,6 millones de personas'      -> (53.6, 'millones de personas', '')
'-0,02%'                         -> (-0.02, '%', '')


In [11]:
parsed = df["Valor"].apply(parse_valor)
df["valor"] = parsed.str[0]
df["unidad"] = parsed.str[1]
df["base"] = parsed.str[2]

# Antes (Valor) y después (valor, unidad, base)
df[["Valor", "valor", "unidad", "base"]].sample(10, random_state=1)

,Valor,valor,unidad,base
40,"4,4%",4.40,%,
31,"-0,02%",-0.02,%,
46,"12,25%",12.25,%,
58,"7,7%",7.70,%,
77,15 % de encuestados,15.00,%,de encuestados
49,"11,9 millones de personas",11.90,millones de personas,
87,"42,3%",42.30,%,
44,2.000.000 COP,2000000.00,COP,
88,"39,9% de oyentes",39.90,%,de oyentes
90,28% de oyentes,28.00,%,de oyentes


In [12]:
df["unidad"].value_counts()

,count
unidad,
%,71
millones,7
millones de personas,3
puntos porcentuales,3
COP,3
Mbps,2
accesos por 100 hab.,2
años,1
municipios,1


---
**3.5. Convertir los periodos en fechas**

---
`Periodo` trae textos como `T2 2026`, `ago-2026`, `ene-sep 2026` o `oct-2024 a oct-2025`. Para usarlos en Power BI necesitamos **fechas reales**.

Creamos tres columnas:
- `tipo_periodo`: mes, trimestre, semestre, año, rango...
- `fecha_inicio` y `fecha_fin`: el primer y el último día del periodo.

`fecha_fin` será la fecha que conectemos con el calendario en Power BI.

In [13]:
MES = {m: i + 1 for i, m in enumerate(["ene", "feb", "mar", "abr", "may", "jun",
                                       "jul", "ago", "sep", "oct", "nov", "dic"])}


def fin_de_mes(anio: int, mes: int) -> pd.Timestamp:
    return pd.Timestamp(anio, mes, 1) + pd.offsets.MonthEnd(0)


def parse_periodo(txt: str):
    """Devuelve (tipo_periodo, fecha_inicio, fecha_fin)."""
    t = txt.lower()
    if t.startswith("n/d"):
        return "no especificado", pd.NaT, pd.NaT
    if m := re.fullmatch(r"(\d{2})/(\d{2})/(\d{4})", t):
        d = pd.Timestamp(int(m[3]), int(m[2]), int(m[1]))
        return "día", d, d
    if m := re.fullmatch(r"t([1-4]) (\d{4})", t):
        q, y = int(m[1]), int(m[2])
        return "trimestre", pd.Timestamp(y, 3 * q - 2, 1), fin_de_mes(y, 3 * q)
    if m := re.fullmatch(r"s([12]) (\d{4})", t):
        s, y = int(m[1]), int(m[2])
        return "semestre", pd.Timestamp(y, 6 * s - 5, 1), fin_de_mes(y, 6 * s)
    if m := re.fullmatch(r"([a-z]{3})-(\d{4})", t):
        y, mes = int(m[2]), MES[m[1]]
        return "mes", pd.Timestamp(y, mes, 1), fin_de_mes(y, mes)
    if m := re.fullmatch(r"([a-z]{3})-([a-z]{3}) (\d{4})", t):
        y = int(m[3])
        return "rango", pd.Timestamp(y, MES[m[1]], 1), fin_de_mes(y, MES[m[2]])
    if m := re.fullmatch(r"([a-z]{3})-(\d{4}) a ([a-z]{3})-(\d{4})", t):
        return ("rango", pd.Timestamp(int(m[2]), MES[m[1]], 1), fin_de_mes(int(m[4]), MES[m[3]]))
    if m := re.fullmatch(r"(\d{4}) vs (\d{4})", t):
        y = int(m[1])
        return "año (comparación)", pd.Timestamp(y, 1, 1), pd.Timestamp(y, 12, 31)
    if m := re.fullmatch(r"(\d{4})", t):
        y = int(m[1])
        return "año", pd.Timestamp(y, 1, 1), pd.Timestamp(y, 12, 31)
    raise ValueError(f"Periodo no reconocido: {txt!r}")


for ejemplo in ["ago-2026", "T2 2026", "ene-sep 2026", "oct-2024 a oct-2025", "N/D (publicado 22/09/2026)"]:
    print(f"{ejemplo!r:32} ->", parse_periodo(ejemplo))

'ago-2026'                       -> ('mes', Timestamp('2026-08-01 00:00:00'), Timestamp('2026-08-31 00:00:00'))
'T2 2026'                        -> ('trimestre', Timestamp('2026-04-01 00:00:00'), Timestamp('2026-06-30 00:00:00'))
'ene-sep 2026'                   -> ('rango', Timestamp('2026-01-01 00:00:00'), Timestamp('2026-09-30 00:00:00'))
'oct-2024 a oct-2025'            -> ('rango', Timestamp('2024-10-01 00:00:00'), Timestamp('2025-10-31 00:00:00'))
'N/D (publicado 22/09/2026)'     -> ('no especificado', NaT, NaT)


In [14]:
per = df["Periodo"].apply(parse_periodo)
df["tipo_periodo"] = per.str[0]
df["fecha_inicio"] = pd.to_datetime(per.str[1])
df["fecha_fin"] = pd.to_datetime(per.str[2])
df["anio_referencia"] = df["fecha_fin"].dt.year.astype("Int64")
df["es_2026"] = (df["anio_referencia"] == 2026).fillna(False).astype(bool)

print("Filas sin fecha (periodo no especificado):", df["fecha_fin"].isna().sum())
df[["Periodo", "tipo_periodo", "fecha_inicio", "fecha_fin"]].drop_duplicates("Periodo").head(12)

Filas sin fecha (periodo no especificado): 6


,Periodo,tipo_periodo,fecha_inicio,fecha_fin
0,oct-2025,mes,2025-10-01,2025-10-31
1,dic-2024 a dic-2025,rango,2024-12-01,2025-12-31
16,ago-2026,mes,2026-08-01,2026-08-31
17,ago-2025,mes,2025-08-01,2025-08-31
20,jul-2026,mes,2026-07-01,2026-07-31
22,sep-2026,mes,2026-09-01,2026-09-30
23,sep-2025,mes,2025-09-01,2025-09-30
25,ene-sep 2026,rango,2026-01-01,2026-09-30
37,T2 2026,trimestre,2026-04-01,2026-06-30
38,S1 2026,semestre,2026-01-01,2026-06-30


---
**3.6. Unificar indicadores que solo cambian por el año**

---

Hay filas como `Tasa de desempleo` y `Tasa de desempleo (mismo mes año anterior)`. Son **el mismo indicador en dos fechas**. Quitamos la aclaración del nombre y dejamos que la **fecha** distinga el año. Así en Power BI se pueden comparar con una sola medida.

In [15]:
filtro = df["Indicador"].str.contains("Tasa de desempleo")
print("ANTES:")
print(df.loc[filtro, ["Indicador", "Periodo"]].to_string(index=False))

ANTES:
                                 Indicador  Periodo
                         Tasa de desempleo ago-2026
Tasa de desempleo (mismo mes año anterior) ago-2025


In [16]:
df["indicador"] = (df["Indicador"]
                   .str.replace(r"\s*\(?mismo mes año anterior\)?", "", regex=True)
                   .str.strip())

print("DESPUÉS:")
print(df.loc[filtro, ["indicador", "valor", "fecha_fin"]].to_string(index=False))

# Control: no puede haber dos filas con el mismo indicador y la misma fecha
repetidos = df.duplicated(["indicador", "fecha_fin"], keep=False) & df["fecha_fin"].notna()
print("\nIndicadores repetidos con la misma fecha (debe ser 0):", repetidos.sum())
assert repetidos.sum() == 0

DESPUÉS:
        indicador  valor  fecha_fin
Tasa de desempleo    9.4 2026-08-31
Tasa de desempleo    8.6 2025-08-31

Indicadores repetidos con la misma fecha (debe ser 0): 0


## 💻Paso 4. Crear la tabla de fuentes

Cada fuente (nombre + tipo + enlace) se repite en muchas filas. La guardamos **una sola vez** en una tabla aparte, `dim_fuente`, con un código (`F01`, `F02`...). En la tabla principal dejamos solo ese código. Esto es lo que Power BI necesita para relacionar tablas.

También agregamos `organismo`: la entidad que origina el dato (DANE, MinTIC...).

In [17]:
ORGANISMOS = [("DANE", "DANE"), ("MinTIC", "MinTIC"), ("Banco de la República", "Banco de la República"),
              ("Reuters", "Reuters Institute"), ("DataReportal", "DataReportal / Kepios"),
              ("CRC", "CRC"), ("Kantar", "Kantar IBOPE"), ("Ibope", "Kantar IBOPE"),
              ("Calculado", "Cálculo propio"), ("Decreto", "Gobierno nacional")]


def organismo(fuente: str) -> str:
    return next((o for clave, o in ORGANISMOS if clave.lower() in fuente.lower()), "Otro")


dim_fuente = (df[["Fuente", "Tipo de fuente", "URL"]].drop_duplicates()
              .rename(columns={"Fuente": "fuente", "Tipo de fuente": "tipo_fuente", "URL": "url"})
              .reset_index(drop=True))
dim_fuente.insert(0, "fuente_id", [f"F{i:02d}" for i in range(1, len(dim_fuente) + 1)])
dim_fuente["organismo"] = dim_fuente["fuente"].map(organismo)

print("Fuentes únicas:", len(dim_fuente))
dim_fuente[["fuente_id", "fuente", "tipo_fuente", "organismo"]].head(10)

Fuentes únicas: 24


,fuente_id,fuente,tipo_fuente,organismo
0,F01,DataReportal / Kepios con datos de la ONU,Reporte privado,DataReportal / Kepios
1,F02,DANE vía Cambio Colombia,Prensa sobre dato oficial,DANE
2,F03,DANE (IPC) vía Vanguardia,Prensa sobre dato oficial,DANE
3,F04,DANE (IPC) vía Bloomberg Línea / Ecosistema St...,Prensa sobre dato oficial,DANE
4,F05,DANE (IPC) vía prensa,Prensa sobre dato oficial,DANE
5,F06,DANE (IPC) vía prensa,Prensa sobre dato oficial,DANE
6,F07,DANE (IPC) vía prensa,Prensa sobre dato oficial,DANE
7,F08,DANE (PIB) vía prensa,Prensa sobre dato oficial,DANE
8,F09,DANE (PIB) vía Safras,Prensa sobre dato oficial,DANE
9,F10,Decreto 1469 de 2025 vía Consultorsalud,Oficial (decreto),Gobierno nacional


## 🧾Paso 5. Armar la tabla principal y guardar

Unimos cada fila con su `fuente_id`, elegimos las columnas finales en un orden claro, le agregamos un `id` y **guardamos los CSV limpios**.

In [18]:
df = df.merge(dim_fuente, left_on=["Fuente", "Tipo de fuente", "URL"],
              right_on=["fuente", "tipo_fuente", "url"], how="left")

fact = df.rename(columns={"Categoria": "categoria", "Subcategoria": "subcategoria",
                          "Periodo": "periodo_original", "Notas": "notas"})
fact = fact[["categoria", "subcategoria", "indicador", "valor", "unidad", "base", "periodo_original",
             "tipo_periodo", "fecha_inicio", "fecha_fin", "anio_referencia", "es_2026",
             "fuente_id", "notas"]]
fact.insert(0, "id", range(1, len(fact) + 1))

fact.to_csv(CLEAN / "fact_indicadores.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
dim_fuente.to_csv(CLEAN / "dim_fuente.csv", index=False, encoding="utf-8-sig")
print("fact_indicadores:", fact.shape, "| dim_fuente:", dim_fuente.shape)
fact.head(8)

fact_indicadores: (95, 15) | dim_fuente: (24, 5)


,id,categoria,subcategoria,indicador,valor,unidad,base,periodo_original,tipo_periodo,fecha_inicio,fecha_fin,anio_referencia,es_2026,fuente_id,notas
0,1,Demografía,Población,Población total,53.6,millones de personas,,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,Estimación ONU
1,2,Demografía,Población,Crecimiento anual de la población,1.0,%,,dic-2024 a dic-2025,rango,2024-12-01,2025-12-31,2025,False,F01,Equivale a +532 mil personas
2,3,Demografía,Población,Población urbana,83.0,%,de la población,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,None
3,4,Demografía,Población,Población rural,17.0,%,de la población,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,None
4,5,Demografía,Sexo,Población femenina,50.7,%,de la población,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,Solo categorías femenino/masculino
5,6,Demografía,Sexo,Población masculina,49.3,%,de la población,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,None
6,7,Demografía,Edad,Edad mediana,32.5,años,,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,None
7,8,Demografía,Edad,Población de 0-4 años,6.6,%,de la población,oct-2025,mes,2025-10-01,2025-10-31,2025,False,F01,Puede no sumar 100 % por redondeo


## 🧼Paso 6 Limpiar la tabla de plataformas

La tabla `plataformas_raw.csv` trae textos como `"31,0 M"`, `"57,9%"`, `"+1,3%"` y valores faltantes como `N/D` o `s/d`. Hacemos lo mismo: quitar espacios, unificar nombres y convertir todo a número.

Además marcamos con `alerta_dato_atipico` las variaciones mayores a ±100 %. La propia fuente advierte que la de Reddit (+217 %) es atípica; **la marcamos en vez de borrarla**.

In [19]:
pl = pd.read_csv(RAW / "plataformas_raw.csv", dtype=str, keep_default_na=False)
print("ANTES:")
pl.head(4)

ANTES:


,Plataforma,Audiencia publicitaria,% poblacion,% internautas,% mujeres,Variacion anual,Periodo variacion,Notas
0,YouTube,"31,0 M","57,9%","74,4%","51,5%","+1,3%",dic-2024 a oct-2025,Alcance publicitario de Google
1,Facebook,"36,4 M","67,9%","87,3 %","50,0%","+0,7%",oct-2024 a oct-2025,Meta cambió su forma de reportar: poca compara...
2,instagram,"21,6 M","40,2%","51,7%","55,0%","+7,7 %",oct-2024 a oct-2025,N/D
3,TikTok,"37,7 M",N/D,"90,5%","43,0%","+24,9%",dic-2024 a oct-2025,"Solo mayores de 18 años (93,0 % de los adultos)"


In [20]:
for col in pl.columns:
    pl[col] = pl[col].str.strip()

NOMBRES = {n.lower(): n for n in ["YouTube", "Facebook", "Instagram", "TikTok", "LinkedIn", "Messenger",
                                  "Snapchat", "Pinterest", "X", "Threads", "Reddit"]}
pl["Plataforma"] = pl["Plataforma"].str.lower().map(NOMBRES)


def a_numero(txt: str):
    """'57,9%' / '+1,3 %' / '31,0 M' -> número; 'N/D' o 's/d' -> vacío."""
    if txt.lower() in {"n/d", "s/d", ""}:
        return float("nan")
    return numero_es(re.sub(r"[%M\s+]", "", txt))


plat = pd.DataFrame({
    "plataforma": pl["Plataforma"],
    "audiencia_publicitaria_millones": pl["Audiencia publicitaria"].map(a_numero),
    "pct_poblacion_total": pl["% poblacion"].map(a_numero),
    "pct_usuarios_internet": pl["% internautas"].map(a_numero),
    "pct_mujeres_audiencia": pl["% mujeres"].map(a_numero),
    "variacion_anual_pct": pl["Variacion anual"].map(a_numero),
    "periodo_variacion": pl["Periodo variacion"].replace("", None),
    "notas": pl["Notas"].where(~pl["Notas"].isin(NULOS), None),
})
plat["alerta_dato_atipico"] = plat["variacion_anual_pct"].abs() > 100
plat.to_csv(CLEAN / "plataformas.csv", index=False, encoding="utf-8-sig")
print("DESPUÉS:")
plat

DESPUÉS:


,plataforma,audiencia_publicitaria_millones,pct_poblacion_total,pct_usuarios_internet,pct_mujeres_audiencia,variacion_anual_pct,periodo_variacion,notas,alerta_dato_atipico
0,YouTube,31.00,57.9,74.4,51.5,1.3,dic-2024 a oct-2025,Alcance publicitario de Google,False
1,Facebook,36.40,67.9,87.3,50.0,0.7,oct-2024 a oct-2025,Meta cambió su forma de reportar: poca compara...,False
2,Instagram,21.60,40.2,51.7,55.0,7.7,oct-2024 a oct-2025,None,False
3,TikTok,37.70,NaN,90.5,43.0,24.9,dic-2024 a oct-2025,"Solo mayores de 18 años (93,0 % de los adultos)",False
4,LinkedIn,18.00,33.6,43.2,49.7,20.0,dic-2024 a dic-2025,"Miembros registrados, no usuarios activos",False
5,Messenger,18.00,33.5,43.1,49.3,-6.8,oct-2024 a oct-2025,Posible efecto de depuración de cuentas,False
6,Snapchat,4.16,7.8,10.0,74.2,-13.0,dic-2024 a oct-2025,None,False
7,Pinterest,10.90,20.4,26.3,68.3,7.6,dic-2024 a oct-2025,Mujeres: sobre audiencia total,False
8,X,5.04,9.4,12.1,33.7,-4.7,dic-2024 a oct-2025,Género inferido por X; fluctúa mucho,False
9,Threads,2.70,5.0,6.5,42.6,NaN,None,Sin dato de variación,False


## Paso 7. Control de calidad final

Antes de dar la limpieza por terminada, repasamos que todo tenga sentido: filas, nulos que quedan (y por qué), y cuántos indicadores son de 2026.

In [21]:
print("Filas finales de indicadores:", len(fact))
print("\nNulos que quedan (esperados):")
print(fact.isna().sum()[lambda s: s > 0].to_string())
print("\nIndicadores por categoría:")
print(fact["categoria"].value_counts().to_string())
print("\nIndicadores con dato de 2026:", int(fact["es_2026"].sum()), "de", len(fact))

Filas finales de indicadores: 95

Nulos que quedan (esperados):
fecha_inicio        6
fecha_fin           6
anio_referencia     6
notas              48

Indicadores por categoría:
categoria
Economía             31
Tecnología           24
Consumo de medios    24
Demografía           16

Indicadores con dato de 2026: 51 de 95


**¿Por qué quedan nulos?**
- `fecha_inicio`, `fecha_fin` y `anio_referencia` (6 filas): la fuente **no especifica el periodo** de esos datos de radio. No inventamos una fecha.
- `notas`: la mayoría de indicadores no traen advertencias.

## Paso 8. Descargar los archivos limpios

Si trabajas en Colab, los archivos se guardan en la carpeta `clean/` de la izquierda y **se borran al cerrar la sesión**. Esta celda los descarga a tu computador. En tu equipo local no hace falta: ya están en `data/clean/`.

In [22]:
try:
    from google.colab import files
    for f in sorted(CLEAN.glob("*.csv")):
        files.download(str(f))
except ImportError:
    print("Archivos guardados en:", CLEAN.resolve())
    print([f.name for f in sorted(CLEAN.glob("*.csv"))])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Listo

El resultado final son **tres tablas limpias:** `fact_indicadores.csv`, `dim_fuente.csv` y `plataformas.csv` para analizar en **Power BI** (ver `docs/guia_power_bi.md`).